Revenging open ai to extract the relationships between the node entities for each article

In [1]:
import os
import certifi
from pymongo import MongoClient
from pymongo.server_api import ServerApi
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Get MongoDB credentials from environment variables
MONGO_URI = os.getenv("MONGO_URI")
DB_NAME = os.getenv("MONGO_DB_NAME")
ARTICLES_COLLECTION_NAME = os.getenv("MONGO_ARTICLES_NAME")
NODES_COLLECTION_NAME = os.getenv("MONGO_NODES_NAME")

# Ensure all necessary environment variables are set
if not all([MONGO_URI, DB_NAME, ARTICLES_COLLECTION_NAME, NODES_COLLECTION_NAME]):
    print("❌ Missing required environment variables. Check your .env file.")
    exit()

# Create MongoDB client with TLS verification
try:
    client = MongoClient(MONGO_URI, server_api=ServerApi('1'), tlsCAFile=certifi.where())
    db = client[DB_NAME]

    # Define collections
    articles_collection = db[ARTICLES_COLLECTION_NAME]  # Stores articles
    nodes_collection = db[NODES_COLLECTION_NAME]  # Stores extracted nodes

    # Verify connection
    client.admin.command('ping')
    print(f"✅ Connected to MongoDB Atlas! Database: {DB_NAME}")

except Exception as e:
    print(f"❌ MongoDB Connection Error: {e}")
    exit()

✅ Connected to MongoDB Atlas! Database: tuone


In [22]:
from openai import OpenAI
import os
from dotenv import load_dotenv
load_dotenv()
openAI_api_key = os.getenv('OPENAI_API_KEY')
client = OpenAI(api_key=openAI_api_key)

def ping_openai(client):
    try:
        response = client.models.list()
        print("✅ Successfully connected to OpenAI API!")
        print(f"Available Models: {[model.id for model in response.data]}")
    except Exception as e:
        print(f"❌ OpenAI API Connection Error: {e}")
ping_openai(client)

✅ Successfully connected to OpenAI API!
Available Models: ['gpt-4.5-preview', 'gpt-4.5-preview-2025-02-27', 'gpt-4o-mini-2024-07-18', 'gpt-4o-mini-audio-preview-2024-12-17', 'o3-mini-2025-01-31', 'dall-e-3', 'dall-e-2', 'o3-mini', 'gpt-4o-audio-preview-2024-10-01', 'gpt-4o-audio-preview', 'gpt-4o-mini-realtime-preview-2024-12-17', 'gpt-4o-mini-realtime-preview', 'o1-mini-2024-09-12', 'o1-mini', 'omni-moderation-latest', 'gpt-4o-mini-audio-preview', 'omni-moderation-2024-09-26', 'whisper-1', 'gpt-4o-realtime-preview-2024-10-01', 'babbage-002', 'chatgpt-4o-latest', 'tts-1-hd-1106', 'gpt-4o-2024-08-06', 'gpt-4o', 'text-embedding-3-large', 'gpt-4o-audio-preview-2024-12-17', 'gpt-4', 'gpt-4o-2024-05-13', 'tts-1-hd', 'o1-preview', 'o1-preview-2024-09-12', 'gpt-4o-2024-11-20', 'gpt-3.5-turbo-instruct-0914', 'gpt-4o-mini-search-preview', 'o1', 'tts-1-1106', 'davinci-002', 'gpt-3.5-turbo-1106', 'gpt-4o-search-preview', 'gpt-4-turbo', 'o1-2024-12-17', 'gpt-4-0125-preview', 'gpt-3.5-turbo-instruc

In [2]:
import json
from bson import json_util
def fetch_articles(limit=00,offset=0):
    try:
        # Fetch articles with an optional limit
        articles_cursor = (articles_collection.find()
                           .skip(offset)
                           .limit(limit))
        articles = list(articles_cursor)

        if articles:
            print(f"✅ Retrieved {len(articles)} articles from MongoDB.\n")
            for idx, article in enumerate(articles, start=1):
                print(f"--- Article {idx} ---")
                # Convert BSON to JSON using json_util
                article_json = json.dumps(article, indent=4, default=json_util.default)
                print(article_json)
        else:
            print("⚠️ No articles found in the collection.")

        return articles

    except Exception as e:
        print(f"❌ Error fetching articles: {e}")
        return []


# Fetch and print articles in JSON format
articles = fetch_articles(limit=1,offset=5)

✅ Retrieved 1 articles from MongoDB.

--- Article 1 ---
{
    "_id": {
        "$oid": "67b1e4200554959fd725edfb"
    },
    "title": "Sweden backs Northvolt battery pilot plant with \u20ac15m",
    "paragraphs": [
        {
            "p1": "Northvolt has found a new backer for setting up a battery cell production pilot plant. The Swedish Energy Agency will support the demonstration line, Northvolt Labs, in V\u00e4ster\u00e5s with a grant of SEK 146m. The new line precedes a planned Swedish Gigafactory.",
            "p2": "Northvolt has been hedging plans to set up a Gigafactory in Sweden for some time. The idea has now got official backing as the Swedish Energy Agency today announced it will support the establishment of Northvolt\u2019s first pilot line with 15 million euros.",
            "p3": "In Kronen that makes 146 millions for the so-called Northvolt Labs. In addition to the demonstration line the new demonstrator facility will also include a research lab, will be used to qu

In [24]:
def read_prompt_from_file_only(file_path):
    with open(file_path, 'r') as file:
        prompt = file.read()
    return prompt

In [25]:
def combine_paragraphs(article):
    paragraphs = article.get('paragraphs', [])
    # Handle missing or empty paragraphs
    if not paragraphs:
        print("⚠️ No paragraphs found in the article.")
        return ""

    combined_text = ""
    for para_obj in paragraphs:
        for key in sorted(para_obj.keys()):
            combined_text += para_obj[key].strip() + " "

    return combined_text.strip()

In [33]:
def extract_nodes(article_id, text, PROMPT_PATH):
    prompt = read_prompt_from_file_only(PROMPT_PATH)

    try:
        completion = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": prompt},
                {"role": "user", "content": f"Here is the article: {text}"}
            ],
            temperature=0
        )

        # Parse the JSON response
        extracted_data = json.loads(completion.choices[0].message.content)
        print(f"✅ Retrieved nodes {extracted_data}\n")

        if "nodes" not in extracted_data:
            raise ValueError("Expected 'nodes' key in LLM output but not found.")

        nodes = extracted_data["nodes"]

        # Format nodes for MongoDB insertion
        formatted_nodes = []
        for node in nodes:
            location_data = node.get("location", {})
            formatted_location = {
                "city": location_data.get("city", ""),
                "country": location_data.get("country", "")
            }

            formatted_node = {
                "article_id": article_id,
                "id": node.get("id"),
                "type": node.get("type"),
                "name": node.get("name"),
                "parent_company": node.get("parent_company"),
                "subsidiaries": node.get("subsidiaries", []),
                "location": formatted_location if location_data else None,  # Only include if present
                "products": node.get("products", []),
                "company": node.get("company", []),  # Ensure list format
                "investment_type": node.get("investment_type"),
                "technology": node.get("technology"),
                "status": node.get("status"),
                "operation_start": node.get("operation_start"),
            }

            # Extract amount details if present
            amount_data = node.get("amount")
            if isinstance(amount_data, dict):
                formatted_node["amount_value"] = amount_data.get("value")
                formatted_node["amount_currency"] = amount_data.get("currency")
                formatted_node["amount_financial_unit"] = amount_data.get("denomination")

            # Extract dates if present
            dates_data = node.get("dates", {})
            if isinstance(dates_data, dict):
                formatted_node["announcement_date"] = dates_data.get("announcement")
                formatted_node["FID"] = dates_data.get("FID")
                formatted_node["construction_start"] = dates_data.get("construction_start")
                formatted_node["operation_start"] = dates_data.get("operation_start")

            formatted_nodes.append(formatted_node)

        return formatted_nodes

    except json.JSONDecodeError as e:
        print(f"❌ JSON Parsing Error: {e}")
        return []
    except Exception as e:
        print(f"❌ Error extracting nodes: {e}")
        return []


In [38]:
def process_articles(limit, offset, PROMPT_PATH):
    # Retrieve articles with limit & offset, sorted by `_id` for consistency
    articles_to_process = list(
        articles_collection.find({}, {"_id": 1, "paragraphs": 1})  # Fetch paragraphs instead of text
        .sort("_id", 1)  # Sort by MongoDB ObjectId (ascending)
        .skip(offset)    # Skip first `offset` articles
        .limit(limit)   # Limit the number of articles
    )

    print(f"🔹 Processing {len(articles_to_process)} articles (Offset: {offset})")

    for article in articles_to_process:
        articleID = str(article["_id"])  # Convert ObjectId to string

        # Extract text from paragraphs
        text = combine_paragraphs(article)

        # Skip if no text is extracted
        if not text:
            print(f"⚠️ No valid text found for Article ID: {articleID}. Skipping.")
            continue

        print(f"📌 Processing Article ID: {articleID}")

        try:
            # Extract and format nodes
            formatted_nodes = extract_nodes(articleID, text, PROMPT_PATH)

            # Ensure extracted nodes are valid before insertion
            if not formatted_nodes:
                print(f"⚠️ No nodes extracted for Article ID: {articleID}. Skipping update.")
                continue

            # Update the article document with extracted nodes and combined text
            update_result = articles_collection.update_one(
                {"_id": article["_id"]},  # Match by article ID
                {"$set": {"combined_text": text, "all_nodes_3": formatted_nodes}}  # Add new fields
            )

            # Log success
            if update_result.modified_count > 0:
                print(f"✅ Updated Article ID: {articleID} with {len(formatted_nodes)} nodes and combined text.")
            else:
                print(f"⚠️ No updates made for Article ID: {articleID}.")

        except Exception as e:
            print(f"❌ Error processing Article ID {articleID}: {e}")

In [39]:
PROMPT_PATH = "node_extraction.txt"
#
process_articles(5,0, PROMPT_PATH)

🔹 Processing 5 articles (Offset: 0)
📌 Processing Article ID: 67b1e41e0554959fd725edf6
✅ Retrieved nodes {'nodes': [{'id': 'mitsubishi_chemical', 'type': 'company', 'name': 'Mitsubishi Chemical', 'HQ_country': 'Japan', 'parent_company': 'null', 'subsidiaries': []}, {'id': 'factory_uk', 'type': 'factory', 'name': 'Mitsubishi Chemical UK Battery Plant', 'location': {'city': 'null', 'country': 'United Kingdom'}, 'technology': 'Battery Electrolyte Manufacturing'}, {'id': 'factory_china', 'type': 'factory', 'name': 'Mitsubishi Chemical China Battery Plant', 'location': {'city': 'null', 'country': 'China'}, 'technology': 'Battery Manufacturing'}]}

✅ Updated Article ID: 67b1e41e0554959fd725edf6 with 3 nodes and combined text.
📌 Processing Article ID: 67b1e41f0554959fd725edf7
✅ Retrieved nodes {'nodes': [{'id': 'byd', 'type': 'company', 'name': 'BYD', 'HQ_country': 'China', 'parent_company': 'null', 'subsidiaries': []}, {'id': 'factory_shanghai', 'type': 'factory', 'name': 'BYD Battery Recycli